In [1]:
import os
import pandas as pd
import numpy as np
import rdkit
from rdkit import Chem
from sklearn.model_selection import train_test_split

import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader

In [2]:
def make_smile_canonical(smile):
    """将 SMILES 转换为标准形式，避免重复"""
    try:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            return np.nan
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return np.nan


In [3]:

def inspect_dataset(df: pd.DataFrame, name: str = "Dataset") -> None:
    """
    打印数据集的基本信息，以便检查：
    - 样本总数
    - 每列缺失值数量
    - 数值列摘要统计
    - SMILES 唯一值比例及最常见项
    - 目标列（Tc, Tg, Density）缺失比例
    """
    total = len(df)
    print(f"=== {name} 概览 ===")
    print(f"样本总数: {total}\n")
    
    # 每列缺失值统计
    print("每列缺失值统计:")
    missing = df.isna().sum().sort_values(ascending=False)
    print(missing[missing > 0], "\n")
    
    # 数值列摘要统计
    print("数值列摘要（包括目标属性）:")
    print(df.describe(include=[float]), "\n")
    
    # SMILES 唯一值比例
    if 'SMILES' in df.columns:
        unique_smiles = df['SMILES'].nunique()
        print(f"SMILES 唯一值: {unique_smiles} / {total} ({unique_smiles/total:.2%})\n")
        print("最常见的 10 个 SMILES 及出现次数:")
        print(df['SMILES'].value_counts().head(10), "\n")
    
    # 目标列缺失比例
    for col in ['Tc', 'Tg', 'Density']:
        if col in df.columns:
            miss = df[col].isna().sum()
            print(f"{col} 缺失: {miss} / {total} ({miss/total:.2%})")
    print("========================\n")

In [4]:
def maybe_correct_offset(df_train, df_extra, target, apply_correction=True):
    """
    如果 train 和 extra 中有重复 SMILES，且都有 target 值，
    则计算系统偏移量（extra - train），并在需要时修正 df_extra 中的 target 值。

    Args:
        df_train (pd.DataFrame): 原始训练数据
        df_extra (pd.DataFrame): 外部增强数据
        target (str): 要增强的目标列名（如 'Tg', 'Tc' 等）
        apply_correction (bool): 是否应用偏移校正

    Returns:
        df_extra_new (pd.DataFrame): 修正后的外部数据副本
    """
    df_train = df_train.copy()
    df_extra = df_extra.copy()

    # Canonical SMILES
    df_train['SMILES'] = df_train['SMILES'].apply(make_smile_canonical)
    df_extra['SMILES'] = df_extra['SMILES'].apply(make_smile_canonical)

    # 保留有 target 的样本
    df_train_tc = df_train[df_train[target].notnull()]
    df_extra_tc = df_extra[df_extra[target].notnull()]

    # 找到重复的 SMILES
    common_smiles = set(df_train_tc['SMILES']) & set(df_extra_tc['SMILES'])
    print(f'common_smiles: {len(common_smiles)}')
    # 提取并聚合
    df_train_common = df_train_tc[df_train_tc['SMILES'].isin(common_smiles)].groupby('SMILES')[target].mean()
    df_extra_common = df_extra_tc[df_extra_tc['SMILES'].isin(common_smiles)].groupby('SMILES')[target].mean()

    # 拼接后对齐
    df_common = pd.concat(
        [df_train_common.rename('train_val'), df_extra_common.rename('extra_val')],
        axis=1
    ).dropna()

    print(f"  → 匹配 SMILES 数量: {len(df_common)}")
    if len(df_common) == 0:
        print(f"  ⚠️ 无法计算 {target} 偏移量，跳过修正")
        return df_extra

    offset = (df_common['extra_val'] - df_common['train_val']).mean()
    std_dev = (df_common['extra_val'] - df_common['train_val']).std()
    print(f"  → {target} 偏移量（extra - train）: {offset:.4f} ± {std_dev:.4f}")

    if apply_correction:
        df_extra[target] = df_extra[target] - offset
        print(f"  ✅ 已对 {target} 进行偏移修正")

    return df_extra

In [5]:

def add_extra_data(df_train, df_extra, target):
    """
    将外部数据 df_extra 根据 target 列添加到 df_train
    1. 标准化 SMILES
    2. 根据 SMILES 聚合外部数据
    3. 填充训练集中缺失的样本
    4. 追加外部独有样本
    """
    print(f"  → 正在增强 {target} 数据，共 {len(df_extra)} 条")
    df_train = df_train.copy()
    df_extra = df_extra.copy()
    df_extra['SMILES'] = df_extra['SMILES'].apply(make_smile_canonical)
    df_extra = df_extra.groupby('SMILES', as_index=False)[target].mean()
    cross_smiles = set(df_extra['SMILES']) & set(df_train['SMILES'])
    print(f'cross_smiles: {len(cross_smiles)}')
    existing = set(df_train[df_train[target].notnull()]['SMILES'])
    cross_smiles -= existing

    for smi in cross_smiles:
        val = df_extra.loc[df_extra['SMILES'] == smi, target].values[0]
        df_train.loc[df_train['SMILES'] == smi, target] = val

    unique_extra = df_extra[~df_extra['SMILES'].isin(df_train['SMILES'])]
    print(f"    填充已有样本 {len(cross_smiles)} 条，新增样本 {len(unique_extra)} 条")
    df_train = pd.concat([df_train, unique_extra], ignore_index=True)
    return df_train

In [6]:
train = pd.read_csv("neurips-open-polymer-prediction-2025/train.csv")

In [7]:
inspect_dataset(train)

=== Dataset 概览 ===
样本总数: 7973

每列缺失值统计:
Tg         7462
Density    7360
Rg         7359
Tc         7236
FFV         943
dtype: int64 

数值列摘要（包括目标属性）:
               Tg          FFV          Tc     Density          Rg
count  511.000000  7030.000000  737.000000  613.000000  614.000000
mean    96.452314     0.367212    0.256334    0.985484   16.419787
std    111.228279     0.029609    0.089538    0.146189    4.608640
min   -148.029738     0.226992    0.046500    0.748691    9.728355
25%     13.674509     0.349549    0.186000    0.890243   12.540328
50%     74.040183     0.364264    0.236000    0.948193   15.052194
75%    161.147595     0.380790    0.330500    1.062096   20.411067
max    472.250000     0.777097    0.524000    1.840999   34.672906 

SMILES 唯一值: 7973 / 7973 (100.00%)

最常见的 10 个 SMILES 及出现次数:
SMILES
*CC(*)c1ccccc1C(=O)OCCCCCC                                                                                                       1
*CC(*)C(=O)OCCCOc1ccc(-c2ccc(C#N)cc2)cc1        

In [8]:
train['SMILES'] = train['SMILES'].apply(make_smile_canonical)

In [9]:
inspect_dataset(train)

=== Dataset 概览 ===
样本总数: 7973

每列缺失值统计:
Tg         7462
Density    7360
Rg         7359
Tc         7236
FFV         943
dtype: int64 

数值列摘要（包括目标属性）:
               Tg          FFV          Tc     Density          Rg
count  511.000000  7030.000000  737.000000  613.000000  614.000000
mean    96.452314     0.367212    0.256334    0.985484   16.419787
std    111.228279     0.029609    0.089538    0.146189    4.608640
min   -148.029738     0.226992    0.046500    0.748691    9.728355
25%     13.674509     0.349549    0.186000    0.890243   12.540328
50%     74.040183     0.364264    0.236000    0.948193   15.052194
75%    161.147595     0.380790    0.330500    1.062096   20.411067
max    472.250000     0.777097    0.524000    1.840999   34.672906 

SMILES 唯一值: 7973 / 7973 (100.00%)

最常见的 10 个 SMILES 及出现次数:
SMILES
*CC(*)c1ccccc1C(=O)OCCCCCC                                                                                                       1
*CC(*)C(=O)OCCCOc1ccc(-c2ccc(C#N)cc2)cc1        

In [10]:
base_path = "neurips-open-polymer-prediction-2025"
extra_dir = os.path.join(base_path, 'smiles-extra-data')

In [11]:
# 2. 增强 Tc
tc_path = os.path.join(base_path, 'Tc_SMILES.csv')
if os.path.exists(tc_path):
    df_tc = pd.read_csv(tc_path).rename(columns={'TC_mean': 'Tc'})
    df_tc = maybe_correct_offset(train, df_tc, 'Tc', apply_correction=False)
    train = add_extra_data(train, df_tc, 'Tc')
else:
    print("  ⚠️ 未找到 Tc 外部数据")

common_smiles: 737
  → 匹配 SMILES 数量: 737
  → Tc 偏移量（extra - train）: -0.0000 ± 0.0003
  → 正在增强 Tc 数据，共 874 条
cross_smiles: 737
    填充已有样本 0 条，新增样本 129 条


In [12]:
tg1_path = os.path.join(extra_dir, 'JCIM_sup_bigsmiles.csv')
if os.path.exists(tg1_path):
    df_tg1 = pd.read_csv(tg1_path, usecols=['SMILES', 'Tg (C)']).rename(columns={'Tg (C)': 'Tg'})
    df_tg1 = maybe_correct_offset(train, df_tg1, 'Tg', apply_correction=False)
    train = add_extra_data(train, df_tg1, 'Tg')
else:
    print("  ⚠️ 未找到 Tg 来源1 数据")

common_smiles: 511
  → 匹配 SMILES 数量: 511
  → Tg 偏移量（extra - train）: 0.0000 ± 0.0000
  → 正在增强 Tg 数据，共 662 条
cross_smiles: 526
    填充已有样本 15 条，新增样本 136 条


In [13]:
tg2_path = os.path.join(extra_dir, 'data_tg3.xlsx')
if os.path.exists(tg2_path):
    df_tg2 = pd.read_excel(tg2_path).rename(columns={'Tg [K]': 'Tg'})
    df_tg2['Tg'] = df_tg2['Tg'] - 273.15
    train = add_extra_data(train, df_tg2, 'Tg')
else:
    print("  ⚠️ 未找到 Tg 来源2 数据")

  → 正在增强 Tg 数据，共 501 条
cross_smiles: 0
    填充已有样本 0 条，新增样本 499 条


In [14]:
d_path = os.path.join(extra_dir, 'data_dnst1.xlsx')
if os.path.exists(d_path):
    df_den = pd.read_excel(d_path)
    df_den = df_den.rename(columns={'density(g/cm3)': 'Density'})[['SMILES', 'Density']]
    df_den['Density'] = pd.to_numeric(df_den['Density'], errors='coerce') - 0.118
    train = add_extra_data(train, df_den, 'Density')
else:
    print("  ⚠️ 未找到 Density 外部数据")

  → 正在增强 Density 数据，共 787 条
cross_smiles: 254
    填充已有样本 110 条，新增样本 525 条


[13:53:23] SMILES Parse Error: syntax error while parsing: *O[Si](*)([R])[R]
[13:53:23] SMILES Parse Error: Failed parsing SMILES '*O[Si](*)([R])[R]' for input: '*O[Si](*)([R])[R]'
[13:53:23] SMILES Parse Error: syntax error while parsing: *NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4
[13:53:23] SMILES Parse Error: Failed parsing SMILES '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4' for input: '*NC(=O)c4ccc3c(=O)n(c2ccc([R]c1ccc(*)cc1)cc2)c(=O)c3c4'
[13:53:23] SMILES Parse Error: syntax error while parsing: O=C=N[R1]N=C=O.O[R2]O.O[R3]O
[13:53:23] SMILES Parse Error: Failed parsing SMILES 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O' for input: 'O=C=N[R1]N=C=O.O[R2]O.O[R3]O'
[13:53:23] SMILES Parse Error: syntax error while parsing: *CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O
[13:53:23] SMILES Parse Error: Failed parsing SMILES '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O' for input: '*CN([R'])Cc2cc([R]c1cc(*)c(O)c(CN([R'])C*)c1)cc(*)c2O'
[13:53:23] SMILES Parse 

In [17]:
# -----------------------------------------------
#  Zero-dependency featurize(): RDKit + Mordred
#  - 取 0/1/2 D 描述符（ignore_3D=True）
#  - 解析失败行 → 全 0     float32 矩阵
# -----------------------------------------------

from rdkit import Chem
from mordred import Calculator, descriptors
import pandas as pd, numpy as np
from tqdm import tqdm
from multiprocessing import cpu_count

# ① 只这一行即可：忽略 3 D 描述符
_calc = Calculator(descriptors, ignore_3D=True)

def _safe_mol(s):
    try:
        return Chem.MolFromSmiles(s)
    except Exception:
        return None

def featurize(smiles, n_jobs: int = 4, verbose: bool = True) -> np.ndarray:
    """
    smiles : pd.Series 或 list[str]
    returns: ndarray (n_samples, n_features)  float32
    """
    smiles = pd.Series(smiles).fillna('').astype(str).str.strip()
    
    # 让 Mordred 自己开多进程：设置环境变量
    import os
    if n_jobs == -1:
        n_jobs = max(1, cpu_count() - 1)
    os.environ["OMP_NUM_THREADS"] = str(n_jobs)
    
    # 1) RDKit Mol
    mols = [_safe_mol(s) for s in tqdm(smiles,
                                       desc="RDKit Mol",
                                       mininterval=1)]
    
    # 2) Mordred → DataFrame
    df = (_calc.pandas(mols)
                .replace([np.inf, -np.inf], np.nan)
                .astype(np.float32)
                .fillna(0.0))
    
    if verbose:
        print(f"✅ featurize: {df.shape[0]} samples × {df.shape[1]} features")
    return df.values

In [20]:
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
import optuna, numpy as np, pandas as pd

storage_uri = "sqlite:///ffv_optuna.db"

X_train = featurize(train['SMILES'])
y_train = train['FFV'].values


print("🔎  X_train 形状:", X_train.shape)
print("🔎  y_train  形状:", y_train.shape)
print("dtype:", X_train.dtype)

print("NaN 计数 ⟶", np.isnan(X_train).sum())
print("Inf 计数 ⟶", np.isinf(X_train).sum())
print("y NaN   ⟶", np.isnan(y_train).sum())

  0%|          | 4/8737 [00:03<2:55:47,  1.21s/it]

/opt/anaconda3/envs/neurips/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/neurips/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  0%|          | 13/8737 [00:05<1:18:01,  1.86it/s]

/opt/anaconda3/envs/neurips/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/opt/anaconda3/envs/neurips/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  0%|          | 25/8737 [00:06<38:07,  3.81it/s]  

...

/opt/anaconda3/envs/neurips/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


  1%|          | 94/8737 [00:18<36:19,  3.97it/s]

/opt/anaconda3/envs/neurips/lib/python3.10/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 11%|█         | 975/8737 [02:40<21:19,  6.07it/s]  


KeyboardInterrupt: 

In [ ]:
# ① 建/载 study （文件自动生成）
study = optuna.create_study(
    study_name="ffv_xgb_search",
    direction="minimize",
    storage=storage_uri,
    load_if_exists=True,          # 已存在就接着跑
)
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
        'max_depth':    trial.suggest_int('max_depth', 4, 10),
        'learning_rate':trial.suggest_float('eta', 1e-3, 0.1, log=True),
        'subsample':    trial.suggest_float('subsample', .5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample', .5, 1.0),
        'reg_lambda':   trial.suggest_float('reg_lambda', 1e-3, 10, log=True)
    }
    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    mae = []
    for train_idx, val_idx in cv.split(X_train):
        model = XGBRegressor(**params)
        model.fit(X_train[train_idx], y_train[train_idx])
        pred = model.predict(X_train[val_idx])
        mae.append(np.abs(pred - y_train[val_idx]).mean())
    return np.mean(mae)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

best_params = study.best_params
model = XGBRegressor(**best_params).fit(X_train, y_train)

In [ ]:
# new_path = os.path.join(base_path, 'train_supplement')
# ffv_path = os.path.join(new_path, 'dataset4.csv')
# if os.path.exists(ffv_path):
#     df_ffv = pd.read_csv(ffv_path)
#     df_ffv = df_ffv.rename(columns={'FFV': 'FFV'})[['SMILES', 'FFV']]
#     df_ffv = maybe_correct_offset(train, df_ffv, 'FFV', apply_correction=False)
#     train = add_extra_data(train, df_ffv, 'FFV')
#     print(f'add dataset4: {len(train)}')    
# else:
#     print("  ⚠️ 未找到 Density 外部数据")

common_smiles: 0
  → 匹配 SMILES 数量: 0
  ⚠️ 无法计算 FFV 偏移量，跳过修正
  → 正在增强 FFV 数据，共 862 条
cross_smiles: 43
    填充已有样本 43 条，新增样本 819 条
add dataset4: 10081


In [24]:
df_t = pd.read_csv('neurips-open-polymer-prediction-2025/train.csv')
df_t = df_t[['SMILES', 'FFV']]
df_t.describe()

,FFV
count,7030.000000
mean,0.367212
std,0.029609
min,0.226992
25%,0.349549
50%,0.364264
75%,0.380790
max,0.777097
